In [9]:
import pandas as pd
import datetime
import time
import os

# ファイル名を'ma_sensor.py'に変更した前提でインポートします
from MA24126A import PowerSensor

# 設定
PORT = "/dev/tty.usbmodem11401" # 実際のCOMポートに合わせて変更してください
BAUDRATE = 19200 # センサーの仕様に合わせて変更
FILENAME = "power_data.csv" # 保存するCSVファイル名

In [10]:
import serial

In [11]:
# センサーオブジェクトの作成と接続
try:
    sensor = PowerSensor(port=PORT, baudrate=BAUDRATE)
    print("センサーに接続しました。")
    
# 必要であれば周波数などを設定（例:10GHz）
    sensor.set_frequency(10)
    
except Exception as e:
    print(f"接続エラー: {e}")
    sensor = None

Connected to /dev/tty.usbmodem11401
センサーに接続しました。
Frequency set to 10.000 GHz


In [12]:
# センサーが生きていればresetとstartをかけます
if sensor:
    print("センサーをリセットします...")
    # リセットコマンド（クラスにないので直接送ります）
    sensor.write("*RST\n")
    time.sleep(2.0) # リセット待ち
    
    print("測定を開始します...")
    sensor.start() # 重要：これがないと止まったままになることがあります
    time.sleep(1.0)
    
    # テスト測定
    test_val = sensor.get_power()
    print(f"現在の値: {test_val} dBm")

センサーをリセットします...
測定を開始します...
Error: パワー値の取得に失敗しました (Response: BAD CMD
OK
-38.570)
現在の値: None dBm


In [13]:
if sensor:
    do_cal = input("ゼロ・キャリブレーションを実行しますか？ (y/n): ")
    
    if do_cal.lower() == 'y':
        print("センサーに信号が入力されていないことを確認してください。")
        input("準備ができたらエンターキーを押してください...")
        
        # クラス内のキャリブレーション機能を呼び出し
        sensor.zero_calibration(show_progress=True)
    else:
        print("キャリブレーションをスキップしました。")

センサーに信号が入力されていないことを確認してください。

Zero calibration start...
Progress: 100%
Zero calibration finished.



In [14]:
data_list = []

print("=== 測定モード ===")
print("エンターキー: 測定実行")
print("'q' + エンター: 終了して保存")
print("==================")

if sensor:
    try:
        while True:
            user_input = input(">> 待機中... (Enterで測定 / qで終了): ")
            
            if user_input.lower() == 'q':
                print("測定を終了します。")
                break
            
            # パワー測定
            power_val = sensor.get_power()
            timestamp = datetime.datetime.now()
            
            if power_val is not None:
                # 画面表示
                print(f"[{timestamp.strftime('%H:%M:%S')}] Power: {power_val:.2f} dBm")
                
                # リストに追加
                data_list.append({
                    "Timestamp": timestamp,
                    "Power_dBm": power_val
                })
            else:
                print("測定エラー: 値が取得できませんでした")

    except KeyboardInterrupt:
        print("\n中断されました。")

=== 測定モード ===
エンターキー: 測定実行
'q' + エンター: 終了して保存
Error: パワー値の取得に失敗しました (Response: OK
-45.798)
測定エラー: 値が取得できませんでした
[11:04:45] Power: -46.46 dBm
[11:04:47] Power: -45.43 dBm
[11:04:48] Power: -48.94 dBm
測定を終了します。


In [7]:
# データフレームに変換してCSV保存
if data_list:
    df = pd.DataFrame(data_list)
    
    # ファイルが既に存在する場合は追記モードにするか、新規作成するか
    # ここでは毎回新規作成または上書きモードで保存します
    df.to_csv(FILENAME, index=False, encoding='utf-8-sig')
    print(f"\nデータを '{FILENAME}' に保存しました。 (件数: {len(df)}件)")
    
    # 最初の5行を表示
    print("\n--- データプレビュー ---")
    print(df.head())
else:
    print("\n保存するデータがありませんでした。")

# 接続終了処理
if sensor:
    sensor.close()
    print("ポートを閉じました。")


保存するデータがありませんでした。
